### 📌 **Seção 1: Introdução ao Problema da Mochila 0-1**
O problema da **mochila 0-1** é um dos problemas clássicos de otimização combinatória. Ele modela diversas situações do mundo real, como **seleção de investimentos, alocação de recursos e planejamento logístico**.

#### **Objetivo**
Dado um conjunto de itens, cada um com um **valor** e um **peso**, queremos selecionar os itens de forma a **maximizar o valor total** sem exceder a **capacidade da mochila**.

#### **Como funciona o problema da mochila?**
- Suponha que temos **4 itens** disponíveis para colocar em uma mochila com capacidade **máxima de 10 kg**:
  1. **Notebook** (peso = 4 kg, valor = R$ 5.000)
  2. **Câmera** (peso = 2 kg, valor = R$ 3.000)
  3. **Smartphone** (peso = 1 kg, valor = R$ 2.000)
  4. **Livro** (peso = 3 kg, valor = R$ 1.500)

- Para cada item, podemos escolher incluí-lo (**1**) ou não (**0**) na mochila. Não é possível levar uma fração de um item (por isso o nome "0-1").

  Se **x_i** representa a escolha do item **i** (1 para incluído, 0 para não incluído), então:

  $$
  w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 \leq 10
  $$

  Onde **w_i** é o peso do item **i**.

#### **Função Objetivo: Maximizando o Valor**
Queremos **maximizar o valor total** dos itens incluídos:

$$
V = v_1 x_1 + v_2 x_2 + v_3 x_3 + v_4 x_4
$$

Onde **v_i** é o valor do item **i**.

---

#### **📊 Exemplo Prático**
Vamos supor que temos os seguintes itens e valores:

| Item       | Peso (kg) | Valor (R$) |
|------------|----------|------------|
| Notebook   | 4        | 5.000      |
| Câmera     | 2        | 3.000      |
| Smartphone | 1        | 2.000      |
| Livro      | 3        | 1.500      |

A mochila tem uma **capacidade máxima de 10 kg**.

##### **🎒 Possíveis soluções**
Vamos avaliar três configurações possíveis:

1️⃣ **Levar todos os itens:**
   - Peso total: **4 + 2 + 1 + 3 = 10 kg** ✅
   - Valor total: **5.000 + 3.000 + 2.000 + 1.500 = R$ 11.500** ✅

2️⃣ **Levar apenas os itens mais valiosos:** (Notebook + Câmera + Smartphone)
   - Peso total: **4 + 2 + 1 = 7 kg** ✅
   - Valor total: **5.000 + 3.000 + 2.000 = R$ 10.000** ✅

3️⃣ **Levar os itens mais leves com alto valor:** (Câmera + Smartphone + Livro)
   - Peso total: **2 + 1 + 3 = 6 kg** ✅
   - Valor total: **3.000 + 2.000 + 1.500 = R$ 6.500** ✅

A **melhor escolha** para este caso seria a primeira opção, pois aproveita a capacidade máxima da mochila e maximiza o valor total.

---

##### **📉 Restrições do Problema**
O problema da mochila 0-1 é caracterizado pelas seguintes restrições:
1. **Cada item pode ser incluído ou não** → (variáveis binárias **0 ou 1**)
2. **A soma dos pesos dos itens escolhidos não pode ultrapassar a capacidade máxima da mochila**
3. **O objetivo é maximizar o valor total dos itens escolhidos**

### 📊 **Seção 2: Resolvendo o problema com o customhys**

#### Instalando o package e baixando do github
 Utilize caso não esteja em um ambiente com o package instalado, como o google colab

In [8]:
# Clonar rep
# !git clone https://github.com/jcrvz/customhys.git .
# Instalar as dependências
# !pip install -r requirements.txt

# Instalar o CUSTOMHyS como pacote
# !pip install .
# %cd customhys
# %cd customhys

In [7]:
import numpy as np

### Lendo Instâncias do problema

In [12]:
def read_knapsack_instances(file_path):
    """
    Lê um arquivo contendo instâncias do problema da mochila 0-1 e retorna uma lista de dicionários.
    
    O arquivo possui o seguinte formato:
      - Primeira linha: nome da instância
      - Linhas de cabeçalho com parâmetros (separados por espaço):
            n <número de itens>
            c <capacidade da mochila>
            z <valor de referência>
            time <tempo>
      - Linhas dos itens (separadas por vírgula):
            <item_index>,<profit>,<weight>,<opt>
      - A linha "-----" indica o fim de uma instância.
    
    :param file_path: Caminho para o arquivo.
    :return: Lista de dicionários, onde cada dicionário representa uma instância.
    """
    instances = []
    with open(file_path, "r") as f:
        current_instance = {}
        items = []
        header_parsed = False  # Indica se já começamos a ler as linhas de itens.
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line == "-----":
                if current_instance:
                    current_instance['items'] = items
                    instances.append(current_instance)
                current_instance = {}
                items = []
                header_parsed = False
                continue
            # Se ainda não estiver lendo itens (header)
            if not header_parsed:
                if "," in line:  # Linha de itens identificada pelo separador vírgula
                    header_parsed = True
                else:
                    # Processa a linha de cabeçalho
                    if "instance_name" not in current_instance:
                        current_instance["instance_name"] = line
                    else:
                        parts = line.split()
                        key = parts[0].lower()
                        if key == "n":
                            current_instance["n"] = int(parts[1])
                        elif key == "c":
                            current_instance["capacity"] = float(parts[1])
                        elif key == "z":
                            current_instance["z"] = float(parts[1])
                        elif key == "time":
                            current_instance["time"] = float(parts[1])
                    continue  # Continua para a próxima linha
            # Se chegou aqui, a linha deve ser de item (ou header já foi finalizado)
            if "," in line:
                parts = line.split(",")
                if len(parts) >= 3:
                    item = {
                        "index": int(parts[0]),
                        "profit": float(parts[1]),
                        "weight": float(parts[2]),
                        "opt": int(parts[3]) if len(parts) >= 4 else None
                    }
                    items.append(item)
        # Finaliza a última instância, se houver
        if current_instance:
            current_instance["items"] = items
            instances.append(current_instance)
    return instances

#### Exemplo de uso

In [4]:
file_path = 'knapPI_1_50_1000.csv'
instances = read_knapsack_instances(file_path)
for inst in instances:
    print("Instância:", inst.get("instance_name"))
    print("  n:", inst.get("n"))
    print("  Capacidade (c):", inst.get("capacity"))
    print("  Valor de referência (z):", inst.get("z"))
    print("  Tempo:", inst.get("time"))
    print("  Itens:")
    for item in inst.get("items", []):
        print(f"    Item {item['index']}: Profit = {item['profit']}, Weight = {item['weight']}, Opt = {item['opt']}")
    print("-" * 40)

Instância: knapPI_1_50_1000_1
  n: 50
  Capacidade (c): 995.0
  Valor de referência (z): 8373.0
  Tempo: 0.0
  Itens:
    Item 1: Profit = 94.0, Weight = 485.0, Opt = 0
    Item 2: Profit = 506.0, Weight = 326.0, Opt = 0
    Item 3: Profit = 416.0, Weight = 248.0, Opt = 0
    Item 4: Profit = 992.0, Weight = 421.0, Opt = 0
    Item 5: Profit = 649.0, Weight = 322.0, Opt = 0
    Item 6: Profit = 237.0, Weight = 795.0, Opt = 0
    Item 7: Profit = 457.0, Weight = 43.0, Opt = 1
    Item 8: Profit = 815.0, Weight = 845.0, Opt = 0
    Item 9: Profit = 446.0, Weight = 955.0, Opt = 0
    Item 10: Profit = 422.0, Weight = 252.0, Opt = 0
    Item 11: Profit = 791.0, Weight = 9.0, Opt = 1
    Item 12: Profit = 359.0, Weight = 901.0, Opt = 0
    Item 13: Profit = 667.0, Weight = 122.0, Opt = 1
    Item 14: Profit = 598.0, Weight = 94.0, Opt = 1
    Item 15: Profit = 7.0, Weight = 738.0, Opt = 0
    Item 16: Profit = 544.0, Weight = 574.0, Opt = 0
    Item 17: Profit = 334.0, Weight = 715.0, Opt =

### Fazendo a função de avaliação

In [ ]:
def knapsack_evaluation(solution, problem):
    """
    Avalia uma solução para o problema da mochila 0-1.
    
    Parâmetros:
      - solution: vetor de decisão contínua gerado pela hyperheurística.
      - problem: dicionário com os dados da instância, contendo:
          • "capacity": capacidade máxima da mochila.
          • "values": lista de lucros/profits dos itens.
          • "weights": lista de pesos dos itens.
    
    Processamento:
      1. Converte a solução para uma solução binária (0 ou 1) usando 0.5 como limiar.
      2. Calcula o peso total e o lucro total dos itens selecionados.
      3. Se o peso total exceder a capacidade, retorna uma penalidade alta.
      4. Se a solução for viável, retorna o negativo do lucro total (para que soluções com maior lucro gerem valores menores, compatíveis com a minimização).
      5. Atualiza a melhor solução global caso o lucro total seja o maior encontrado até o momento.
    
    Retorno:
      - Um valor float que representa a função de avaliação (menor é melhor).
    """
    global best_solution, best_profit
        
    # Converte a solução contínua para uma solução binária (0 ou 1)
    solution = np.array(solution)
    binary_solution = np.where(solution < 0.5, 0, 1)
    
    # Calcula o peso total e o lucro total dos itens selecionados
    total_weight = np.dot(binary_solution, np.array(problem["weights"]))
    total_profit = np.dot(binary_solution, np.array(problem["values"]))
    
    # Se a solução for inviável (peso excede a capacidade), aplica uma penalidade alta
    if total_weight > problem["capacity"]:
        return float(score)

    else:
        # Solução viável: o objetivo é maximizar o lucro, portanto retornamos seu negativo
        score = -total_profit
        # Atualiza a melhor solução global, se o lucro atual for o melhor encontrado
        if total_profit > best_profit:
            best_profit = total_profit
            best_solution = binary_solution.copy()
    
    return float(score)


### Configurando problema

In [8]:
def configure_knapsack_problem(knapsack_instance):
    """
    Gera um dicionário de configuração para a Hyper-Heurística com base em uma instância do problema da mochila 0-1.
    
    A instância 'knapsack_instance' deve conter os seguintes campos:
      - "capacity": Capacidade máxima da mochila.
      - "items": Lista de itens, onde cada item é um dicionário com as chaves "profit" e "weight".
      - "n": (Opcional) Número de itens. Se não existir, usa o tamanho da lista "items".
    
    A função de avaliação utilizada é a 'knapsack_evaluation'.
    Os limites (boundaries) para cada item variam entre 0 e 1.
    
    :param knapsack_instance: Dicionário com os dados da instância da mochila.
    :return: Dicionário formatado para a Hyper-Heurística.
    """
    # Define o número de itens: usa o campo "n" se existir, ou a quantidade de itens na lista.
    n_items = knapsack_instance.get("n", len(knapsack_instance.get("items", [])))
    
    # Cria um dicionário com os dados necessários para a avaliação.
    problem_data = {
        "capacity": knapsack_instance["capacity"],
        "values": [item["profit"] for item in knapsack_instance["items"]],
        "weights": [item["weight"] for item in knapsack_instance["items"]],
    }
    
    return {
        "function": lambda solution: knapsack_evaluation(solution, problem_data),
        "is_constrained": True,
        "boundaries": ([0] * n_items, [1] * n_items)
    }

### Configuração de hyperheuristica

In [11]:
# 🔹 Configuração dos parâmetros da Hyper-Heurística
hh_config = {
    "cardinality": 10,  # Máximo de operadores na sequência
    "cardinality_min": 1,  # Mínimo de operadores na sequência
    "num_iterations": 100,  # Número de iterações
    "num_agents": 30,  # Tamanho da população
    "as_mh": False,  # Usa a sequência de HH como uma metaheurística completa?
    "num_replicas": 20,  # Número de réplicas por MH
    "num_steps": 100,  # Número de tentativas por passo da HH
    "stagnation_percentage": 0.40,  # Percentual de estagnação para controle
    "max_temperature": 1,  # Temperatura inicial (Simulated Annealing)
    "min_temperature": 1e-6,  # Temperatura mínima (Simulated Annealing)
    "cooling_rate": 1e-3,  # Taxa de resfriamento (Simulated Annealing)
    "temperature_scheme": "fast",  # Esquema de temperatura
    "acceptance_scheme": "exponential",  # Critério de aceitação de soluções
    "allow_weight_matrix": True,  # Permite matriz de pesos
    "trial_overflow": False,  # Política de overflow de tentativas
    "repeat_operators": True,  # Permite repetição de operadores na sequência
    "verbose": True,  # Exibir logs e progresso
    "learning_portion": 0.5,  # Percentual de aprendizado das sequências
    "solver": "static",  # Tipo de solver
}

### Rodando hyperheuristica

In [ ]:
from customhys.hyperheuristic import Hyperheuristic

# 🔹 Criar a instância da Hyper-Heurística
hh = Hyperheuristic(heuristic_space='default.txt', problem=configure_knapsack_problem(instances[0]), parameters=hh_config)

# 🔹 Executar a otimização
print("\n🚀 [LOG] Iniciando execução da Hyper-Heurística...\n")

result = hh.solve()  # Executa a otimização

print("\n🔍 [LOG] Resultado da Hyper-Heurística:", result)